# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [83]:
#Libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import train_test_split

# Banging_Clasificattion
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Forest_Clasificattion
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Gradiente_Boosting_Clasificación
from sklearn.ensemble import GradientBoostingClassifier

# AdaBoost_Classifier
from sklearn.ensemble import AdaBoostClassifier

In [84]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [85]:
spaceship.shape

(8693, 14)

**Check for data types**

In [86]:
spaceship.dtypes

PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object

**Check for missing values**

In [87]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [88]:
spaceship = spaceship.dropna()
spaceship.shape


(6606, 14)

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [89]:
spaceship['Cabin'].head()
spaceship["Cabin"] = spaceship["Cabin"].str[0]
spaceship["Cabin"].unique() #para verificar. Vemos que salen esos valores

<StringArray>
['B', 'F', 'A', 'G', 'E', 'C', 'D', 'T']
Length: 8, dtype: str

- Drop PassengerId and Name

In [90]:
spaceship = spaceship.drop(columns=["PassengerId", "Name"])
print(spaceship.shape)
spaceship.head()


(6606, 12)


,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


 For non-numerical columns, do dummies.  - Lo que llama One-Hot-Encoding


In [91]:
spaceship = pd.get_dummies(spaceship)
print(spaceship.shape)
spaceship.head()

(6606, 25)


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,False,True,False,...,False,False,False,False,False,False,False,True,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,True,False,False,...,False,False,True,False,False,False,False,True,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,False,True,False,...,False,False,False,False,False,False,False,True,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,True,True,False,False,...,False,False,True,False,False,False,False,True,True,False


---

- Feature Selection  → seleccionar las mejores columnas (features)

In [92]:
features = spaceship.drop(columns=["Transported"]) #borramos Transported de la "asignacion" por que va a ser el objetivo
                                                    # no lo borramos del da
target = spaceship["Transported"] # asignamos esa columna al objetivo

**Perform Train Test Split**

In [93]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.20, random_state=0
)

X_train.shape, X_test.shape

((5284, 24), (1322, 24))

Now perform the same as before:
- Feature Scaling  → escalar/normalizar las variables numéricas

In [94]:
normalizer = MinMaxScaler()  

In [95]:
normalizer.fit(X_train) # Calcula X_min y X_max de cada columna usando solo los datos de entrenamiento.

,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False


In [96]:
X_train_norm = normalizer.transform(X_train) # Para que todas las columnas estén en la misma escala (0–1).
                                                # valores pequeños → cerca de 0
                                                # valores grandes → cerca de 1
X_test_norm = normalizer.transform(X_test) # Normaliza el test usando los mismos valores del train.

In [97]:
X_train_norm = pd.DataFrame(X_train_norm, columns = X_train.columns) # Devuelve los nombres de columnas.
X_test_norm = pd.DataFrame(X_test_norm, columns = X_test.columns)

In [98]:
X_train.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True
3432,32.0,0.0,0.0,0.0,0.0,0.0,False,False,True,False,...,False,True,False,False,False,False,False,True,True,False
7312,4.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False,...,False,False,False,True,False,False,False,True,True,False
2042,30.0,0.0,236.0,0.0,1149.0,0.0,True,False,False,True,...,False,False,True,False,False,False,False,True,True,False
4999,17.0,13.0,0.0,565.0,367.0,1.0,False,False,True,True,...,False,True,False,False,False,False,False,True,True,False
5755,26.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False,...,False,False,False,True,False,False,False,True,True,False


---

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

Bagging involves training multiple instances of the same base model on different subsets of the training data. The final prediction is obtained by averaging or voting over predictions from these models.

Just for baseline, our current best model is a Decision Tree with R-Squared of 0.70, lets see how ensembles works

In [99]:


bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),  # árbol base
    n_estimators=200,                    # número de árboles
    max_samples=0.8,                     # porcentaje de muestras por árbol
    bootstrap=True,                      # Bagging (si fuera False → Pasting)
    random_state=0
)


Training Bagging model with our normalized data

In [100]:
# Entrenamos
bag_clf.fit(X_train_norm, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeClassifier()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",200
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",0.8
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",0
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


Evaluate model's performance

In [101]:
pred_bag_clf = bag_clf.predict(X_test_norm)


# Métricas 
print("Accuracy :", accuracy_score(y_test, pred_bag_clf))
print("Precision:", precision_score(y_test, pred_bag_clf))
print("Recall   :", recall_score(y_test, pred_bag_clf))
print("F1-score :", f1_score(y_test, pred_bag_clf))


Accuracy : 0.791981845688351
Precision: 0.7915407854984894
Recall   : 0.7927382753403933
F1-score : 0.7921390778533636


---

- Random Forests

In [102]:
# Modelo Random Forest para CLASIFICACIÓN
rf_clf = RandomForestClassifier(
    n_estimators=100,   # número de árboles
    max_depth=20,       # profundidad máxima
    random_state=0
)

In [103]:
# Entrenamos
rf_clf.fit(X_train_norm, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [104]:
# Predecimos
pred_rf_clf = rf_clf.predict(X_test_norm)

In [105]:
# Métricas 
print("Accuracy :", accuracy_score(y_test, pred_rf_clf))
print("Precision:", precision_score(y_test, pred_rf_clf))
print("Recall   :", recall_score(y_test, pred_rf_clf))
print("F1-score :", f1_score(y_test, pred_rf_clf))

Accuracy : 0.7957639939485628
Precision: 0.7993874425727412
Recall   : 0.789712556732224
F1-score : 0.7945205479452054


- Gradient Boosting

In [106]:
# Modelo Gradient Boosting para CLASIFICACIÓN
gb_clf = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=0
)

In [107]:
# Entrenamos
gb_clf.fit(X_train_norm, y_train)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",300
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``,

In [108]:
# Predecimos
pred_gb_clf = gb_clf.predict(X_test_norm)

In [109]:
# Métricas
print("Accuracy :", accuracy_score(y_test, pred_gb_clf))
print("Precision:", precision_score(y_test, pred_gb_clf))
print("Recall   :", recall_score(y_test, pred_gb_clf))
print("F1-score :", f1_score(y_test, pred_gb_clf))

Accuracy : 0.7866868381240545
Precision: 0.7628294036061026
Recall   : 0.8320726172465961
F1-score : 0.7959479015918958


- Adaptive Boosting

In [110]:
#your code here# Modelo AdaBoost para CLASIFICACIÓN
ada_clf = AdaBoostClassifier(
    n_estimators=300,
    learning_rate=0.05,
    random_state=0
)

In [111]:
# Entrenamos
ada_clf.fit(X_train_norm, y_train)

,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",300
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",0.05
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",0


In [112]:
# Predecimos
pred_ada_clf = ada_clf.predict(X_test_norm)

In [113]:
# Métricas
print("Accuracy :", accuracy_score(y_test, pred_ada_clf))
print("Precision:", precision_score(y_test, pred_ada_clf))
print("Recall   :", recall_score(y_test, pred_ada_clf))
print("F1-score :", f1_score(y_test, pred_ada_clf))

Accuracy : 0.7556732223903178
Precision: 0.8061594202898551
Recall   : 0.6732223903177005
F1-score : 0.7337180544105524


Which model is the best and why?

In [114]:
resultados = {
    "Modelo": ["Bagging", "Random Forest", "Gradient Boosting", "AdaBoost"],
    "Accuracy": [
        accuracy_score(y_test, pred_bag_clf),
        accuracy_score(y_test, pred_rf_clf),
        accuracy_score(y_test, pred_gb_clf),
        accuracy_score(y_test, pred_ada_clf)
    ],
    "Precision": [
        precision_score(y_test, pred_bag_clf),
        precision_score(y_test, pred_rf_clf),
        precision_score(y_test, pred_gb_clf),
        precision_score(y_test, pred_ada_clf)
    ],
    "Recall": [
        recall_score(y_test, pred_bag_clf),
        recall_score(y_test, pred_rf_clf),
        recall_score(y_test, pred_gb_clf),
        recall_score(y_test, pred_ada_clf)
    ],
    "F1-score": [
        f1_score(y_test, pred_bag_clf),
        f1_score(y_test, pred_rf_clf),
        f1_score(y_test, pred_gb_clf),
        f1_score(y_test, pred_ada_clf)
    ]
}

tabla_resultados = pd.DataFrame(resultados).sort_values(by="F1-score", ascending=False)
tabla_resultados



,Modelo,Accuracy,Precision,Recall,F1-score
2,Gradient Boosting,0.786687,0.762829,0.832073,0.795948
1,Random Forest,0.795764,0.799387,0.789713,0.794521
0,Bagging,0.791982,0.791541,0.792738,0.792139
3,AdaBoost,0.755673,0.806159,0.673222,0.733718


Conclusiones:

 El mejor modelo es GradientBoosting.
 Tiene la mejor nota final (F1-score), que es la métrica más importante,  porque combina Precision y Recall y refleja el rendimiento real del modelo.

 Eliminamos primero a **AdaBoost**:
 - Tiene la peor nota en F1-score.
 - Tiene la peor nota en Recall.
 - Y un modelo con Recall bajo NO sirve:
     → El modelo se deja fuera muchos casos que sí eran positivos.
     → En este tipo de problemas , los positivos perdidos son vidas.

 Random Forest queda en segundo lugar:
 - Muy equilibrado.
 - Muy buena Precision y Recall.
 - F1 muy cercana a GradientBoosting.

 Bagging queda en tercer lugar:
 - Correcto, estable, pero sin destacar.
 - F1 media-alta, pero inferior a los dos mejores.

 Conclusión final:
 GradientBoosting es el mejor modelo porque logra el mejor equilibrio
 entre Precision y Recall, obteniendo la mejor nota final (F1-score).




